### Implement ReAct with LangGraph

### ReAct
- Agent design technique that enables an LLM to think step-by-step, use tools, observe results, and then continue reasoning before generating the final response.

- create_agent is a prebuilt LangGraph helper that creates a ReAct-style agent capable of reasoning, selecting tools, executing them, observing results, and producing a final answer. 
- It abstracts away the graph construction while still leveraging LangGraph's execution engine.

In [ ]:
import os
from  typing import List , Annotated, TypedDict, Sequence
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage


In [ ]:
### 1. Document Processing

urls=[
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/"
]

loaders = [WebBaseLoader(url) for url in urls]

docs=[]

for loader in loaders:
    docs.extend(loader.load())

docs

In [ ]:
## 2. Recursive character txt splitter as vector store

splitter =RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(split_docs, embedding)
retriever= vectorstore.as_retriever()

retriever.invoke("what is reward function") # test the retriever

In [ ]:
from langchain_core.tools import tool

@tool
def retriever_tool_func(query:str)->str:
    """Use this tool to fetch the relevant knowledge base info"""
    docs=retriever.invoke(query)
    return "\n".join([doc.pagecontent for doc in docs])

In [ ]:
api_wrapper_arxiv = WikipediaAPIWrapper(
    top_k_results=2,
    doc_content_char_limit=500
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper_arxiv)



In [ ]:
### Combine all the tools
tools =[wiki_tool,retriever_tool_func]

In [ ]:
# Intialize chat model

from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="qwen/qwen3-32b")



In [ ]:
# create the native Langgraph react agent
from langchain.agents import create_agent

agent_node = create_agent( # internally constructs state graph
    model=llm_groq,
    tools=tools
)

agent_node

In [ ]:
# Create agent State


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages] # messages is a list-like collection of LangChain message objects.[HumanMessage(...), AIMessage(...),ToolMessage(...)]

In [ ]:
# Build the graph

builder=StateGraph(AgentState)

builder.add_node("react_agent", agent_node)

builder.add_edge(START, "react_agent")
builder.add_edge("react_agent", END)

graph = builder.compile()
graph

In [ ]:
# Run the ReAct AgentState

if __name__=="__main__":
    user_query="what is reeward hacking and dies how wikipedia describe it?"
    state= {"messages": [HumanMessage(content=user_query)]}
    result = graph.invoke(state)

    print(result['messages'][-1].content)